In [1]:
!pip install -q --upgrade numpy
import importlib, sys

# Remove any cached numpy modules
mods_to_remove = [m for m in sys.modules if 'numpy' in m]
for m in mods_to_remove:
    del sys.modules[m]

In [5]:
!pip install -q --upgrade \
  "numpy" \
  "pandas" \
  "datasets" \
  "transformers" \
  "evaluate" \
  "accelerate" \
  "scikit-learn" \
  "sentencepiece"

In [2]:
!pip install -q --force-reinstall --no-deps \
  "torch" \
  "torchvision" \
  "torchaudio"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 155.9 MB/s eta 0:00:00


In [1]:
!pip show numpy torch transformers

Name: numpy
Version: 2.4.4
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: accelerate, access, albucore, albumentations, ale-py, arviz, astropy, autograd, bigframes, blis, blosc2, bokeh, Bottleneck, bqplot, clarabel, cmdstanpy, contourpy, cucim-cu12, cuda-core, cudf-cu12, cufflinks, cuml-cu12, cupy-cuda12x, cuvs-cu12, cvxpy, cyipopt, dask-cuda, dask-cudf-cu12, datasets, db-dtypes, diffusers, dm-tree, dopamine_rl, esda, evaluate, flax, folium, geemap, geopandas, gradio, grain, gym, gymnasium, h5netcdf, h5py, hdbscan, highspy, holoviews, hyperopt, ImageIO, imbalanced-learn, inequality, jax, jaxlib, keras, keras-hub, libpysal, librosa, lightgbm, mapclassify, matplotlib, matplotlib-venn, mgwr, missingno, mizani, ml_dtypes, mlxtend, moviepy, music21, nibabel, numba, numexpr, nx-cugraph-cu12, opencv-contrib-python, opencv

In [2]:
import numpy as np
import pandas as pd
import torch

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

numpy: 2.4.4
pandas: 3.0.2
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
import random

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import evaluate

In [4]:
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

import ctypes
ctypes.CDLL("/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0")
print("loaded successfully")

loaded successfully


In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [6]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

dialog: a list of string features.
act: a list of classification labels, with possible values including __dummy__ (0), inform (1), question (2), directive (3) and commissive (4).
emotion: a list of classification labels, with possible values including no emotion (0), anger (1), disgust (2), fear (3), happiness (4), sadness (5) and surprise (6).

In [7]:
ds = load_dataset("pixelsandpointers/better_daily_dialog")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json:   0%|          | 0.00/995 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.66M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/339k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/336k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87170 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8069 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7740 [00:00<?, ? examples/s]

In [8]:
print(ds)
print(ds["train"][0])
print(ds["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['dialog_id', 'utterance', 'turn_type', 'emotion'],
        num_rows: 87170
    })
    validation: Dataset({
        features: ['dialog_id', 'utterance', 'turn_type', 'emotion'],
        num_rows: 8069
    })
    test: Dataset({
        features: ['dialog_id', 'utterance', 'turn_type', 'emotion'],
        num_rows: 7740
    })
})
{'dialog_id': 0, 'utterance': 'Say , Jim , how about going for a few beers after dinner ? ', 'turn_type': 3, 'emotion': 0}
['dialog_id', 'utterance', 'turn_type', 'emotion']


In [9]:
def flatten_dailydialog(split):
    rows = {
        "utterance": [],
        "context_2": [],
        "context_3": [],
        "label": [],
        "emotion": [],
        "dialog_id": [],
        "turn_id": [],
    }

    from itertools import groupby
    keyfunc = lambda x: x["dialog_id"]
    sorted_split = sorted(split, key=keyfunc)

    for dialog_id, turns in groupby(sorted_split, key=keyfunc):
        turns = list(turns)
        utterances = [t["utterance"] for t in turns]
        acts = [t["turn_type"] for t in turns]   # 1-4
        emotions = [t["emotion"] for t in turns] # should already be 0-6

        for i, utt in enumerate(utterances):
            current_speaker = "[S1]" if i % 2 == 0 else "[S2]"

            prev2 = []
            for j in range(max(0, i - 2), i):
                spk = "[S1]" if j % 2 == 0 else "[S2]"
                prev2.append(f"{spk} {utterances[j]}")
            context_2 = " ".join(prev2 + [f"{current_speaker} {utt}"])

            prev3 = []
            for j in range(max(0, i - 3), i):
                spk = "[S1]" if j % 2 == 0 else "[S2]"
                prev3.append(f"{spk} {utterances[j]}")
            context_3 = " ".join(prev3 + [f"{current_speaker} {utt}"])

            rows["utterance"].append(utt)
            rows["context_2"].append(context_2)
            rows["context_3"].append(context_3)
            rows["label"].append(acts[i] - 1)     # act: 1-4 -> 0-3
            rows["emotion"].append(int(emotions[i]))  # emotion: keep as 0-6
            rows["dialog_id"].append(dialog_id)
            rows["turn_id"].append(i)

    return Dataset.from_dict(rows)

flat_ds = DatasetDict({
    "train": flatten_dailydialog(ds["train"]),
    "validation": flatten_dailydialog(ds["validation"]),
    "test": flatten_dailydialog(ds["test"]),
})

In [10]:
id2label = {
    0: "inform",
    1: "question",
    2: "directive",
    3: "commissive"
}
label2id = {v: k for k, v in id2label.items()}
print(id2label)

{0: 'inform', 1: 'question', 2: 'directive', 3: 'commissive'}


In [11]:
for split_name in ["train", "validation", "test"]:
    counts = pd.Series(flat_ds[split_name]["label"]).value_counts().sort_index()
    print(f"\n{split_name.upper()}")
    for i, count in counts.items():
        print(f"{id2label[i]}: {count}")


TRAIN
inform: 39873
question: 24974
directive: 14242
commissive: 8081

VALIDATION
inform: 3125
question: 2244
directive: 1775
commissive: 925

TEST
inform: 3534
question: 2210
directive: 1278
commissive: 718


In [12]:
X_train = flat_ds["train"]["utterance"]
y_train = flat_ds["train"]["label"]

X_val = flat_ds["validation"]["utterance"]
y_val = flat_ds["validation"]["label"]

X_test = flat_ds["test"]["utterance"]
y_test = flat_ds["test"]["label"]

print("Train examples:", len(X_train))
print("Validation examples:", len(X_val))
print("Test examples:", len(X_test))

Train examples: 87170
Validation examples: 8069
Test examples: 7740


In [13]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    lowercase=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (87170, 20000)
Validation TF-IDF shape: (8069, 20000)
Test TF-IDF shape: (7740, 20000)


In [14]:
logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

logreg.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [15]:
val_preds = logreg.predict(X_val_tfidf)
test_preds = logreg.predict(X_test_tfidf)

print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Validation Macro F1:", f1_score(y_val, val_preds, average="macro"))

print("\nTest Accuracy:", accuracy_score(y_test, test_preds))
print("Test Macro F1:", f1_score(y_test, test_preds, average="macro"))

Validation Accuracy: 0.7290866278349237
Validation Macro F1: 0.7021684685183232

Test Accuracy: 0.739405684754522
Test Macro F1: 0.6909947509297478


In [16]:
print(classification_report(
    y_test,
    test_preds,
    target_names=[id2label[i] for i in range(4)]
))

              precision    recall  f1-score   support

      inform       0.83      0.71      0.77      3534
    question       0.89      0.83      0.86      2210
   directive       0.61      0.72      0.66      1278
  commissive       0.38      0.61      0.47       718

    accuracy                           0.74      7740
   macro avg       0.68      0.72      0.69      7740
weighted avg       0.77      0.74      0.75      7740



In [17]:
cm = confusion_matrix(y_test, test_preds)
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{id2label[i]}" for i in range(4)],
    columns=[f"pred_{id2label[i]}" for i in range(4)]
)
cm_df

,pred_inform,pred_question,pred_directive,pred_commissive
true_inform,2524,115,349,546
true_question,180,1840,143,47
true_directive,160,83,919,116
true_commissive,170,19,89,440


In [18]:
error_df = pd.DataFrame({
    "utterance": X_test,
    "true_label": [id2label[y] for y in y_test],
    "pred_label": [id2label[p] for p in test_preds]
})

errors = error_df[error_df["true_label"] != error_df["pred_label"]].copy()
errors.head(25)

,utterance,true_label,pred_label
2,"Weed ! You know ? Pot , Ganja , Mary Jane som...",directive,inform
4,I also have blow if you prefer to do a few li...,directive,inform
10,Yeah ?,question,inform
20,"Well , we use the exhaust gases from our prin...",inform,directive
28,"Yes , I believe there are green teas , black ...",question,inform
32,"Sure , I like drinking tea at teahouses .",inform,commissive
33,"Oh , so do I .",inform,question
44,"Yes , I like travelling . I am young , and un...",inform,commissive
50,"ok . You haven ’ t seen my company car , have...",directive,question
51,no . let me take a look ... it ’ s longer tha...,commissive,inform


In [19]:
checkpoint = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
def tokenize_utterance(batch):
    return tokenizer(
        batch["utterance"],
        truncation=True,
        max_length=96
    )

tok_utt = flat_ds.map(tokenize_utterance, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "label"]

tok_utt = tok_utt.remove_columns(
    [c for c in tok_utt["train"].column_names if c not in cols_to_keep]
)

tok_utt = tok_utt.rename_column("label", "labels")
tok_utt.set_format("torch")

Map:   0%|          | 0/87170 [00:00<?, ? examples/s]

Map:   0%|          | 0/8069 [00:00<?, ? examples/s]

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

In [21]:
model_utt = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./roberta_utt_debug",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=False,
    bf16=False,
    report_to="none"
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_utt = Trainer(
    model=model_utt,
    args=training_args,
    train_dataset=tok_utt["train"],
    eval_dataset=tok_utt["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_utt.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.513182,0.524766,0.797249,0.747152
2,0.426097,0.505132,0.812864,0.760600
3,0.359310,0.522354,0.814351,0.765694


TrainOutput(global_step=16347, training_loss=0.4328630157674191, metrics={'train_runtime': 539.2746, 'train_samples_per_second': 484.929, 'train_steps_per_second': 30.313, 'total_flos': 5811090839926944.0, 'train_loss': 0.4328630157674191, 'epoch': 3.0})

In [23]:
from collections import Counter

pred_output = trainer_utt.predict(tok_utt["validation"])
preds = pred_output.predictions.argmax(axis=-1)

print("Predicted label counts:", Counter(preds.tolist()))
print("True label counts:", Counter(list(tok_utt["validation"]["labels"])))

Predicted label counts: Counter({0: 3284, 1: 2322, 2: 1760, 3: 703})
True label counts: Counter({tensor(1): 1, tensor(0): 1, tensor(2): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(1): 1, tensor(2): 1, tensor(1): 1, tensor(2): 1, tensor(3): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(2): 1, tensor(1): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(2): 1, tensor(1): 1, tensor(0): 1, tensor(2): 1, tensor(3): 1, tensor(2): 1, tensor(3): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(2): 1, tensor(3): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(2): 1, tensor(2): 1, tensor(1): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(0): 1, tensor(

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = list(tok_utt["validation"]["labels"])
y_pred = preds.tolist()

print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(4)],
    digits=4
))

print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

      inform     0.8072    0.8483    0.8273      3125
    question     0.9212    0.9532    0.9369      2244
   directive     0.7625    0.7561    0.7593      1775
  commissive     0.6245    0.4746    0.5393       925

    accuracy                         0.8144      8069
   macro avg     0.7789    0.7580    0.7657      8069
weighted avg     0.8081    0.8144    0.8098      8069

[[2651   49  229  196]
 [  13 2139   88    4]
 [ 256  113 1342   64]
 [ 364   21  101  439]]


BERTweet

In [25]:
checkpoint_bt = "vinai/bertweet-base"

# BERTweet often works more smoothly with use_fast=False
tokenizer_bt = AutoTokenizer.from_pretrained(checkpoint_bt, use_fast=False)

def tokenize_utterance_bt(batch):
    return tokenizer_bt(
        batch["utterance"],
        truncation=True,
        max_length=96
    )

tok_utt_bt = flat_ds.map(tokenize_utterance_bt, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "label"]

tok_utt_bt = tok_utt_bt.remove_columns(
    [c for c in tok_utt_bt["train"].column_names if c not in cols_to_keep]
)

tok_utt_bt = tok_utt_bt.rename_column("label", "labels")
tok_utt_bt.set_format("torch")

model_utt_bt = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_bt,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics_bt(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

data_collator_bt = DataCollatorWithPadding(tokenizer=tokenizer_bt)

training_args_bt = TrainingArguments(
    output_dir="./bertweet_utt_debug",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=False,
    bf16=False,
    report_to="none"
)


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


...

Map:   0%|          | 0/8069 [00:00<?, ? examples/s]

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initi

In [26]:
trainer_utt_bt = Trainer(
    model=model_utt_bt,
    args=training_args_bt,
    train_dataset=tok_utt_bt["train"],
    eval_dataset=tok_utt_bt["validation"],
    data_collator=data_collator_bt,
    compute_metrics=compute_metrics_bt
)

trainer_utt_bt.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.501463,0.514303,0.802454,0.750190
2,0.408338,0.496334,0.815343,0.765215
3,0.338144,0.524561,0.816706,0.765879


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

TrainOutput(global_step=16347, training_loss=0.4159817228027314, metrics={'train_runtime': 541.1367, 'train_samples_per_second': 483.26, 'train_steps_per_second': 30.209, 'total_flos': 5564835025677408.0, 'train_loss': 0.4159817228027314, 'epoch': 3.0})

In [27]:
pred_output_bt = trainer_utt_bt.predict(tok_utt_bt["validation"])
preds_bt = pred_output_bt.predictions.argmax(axis=-1)

from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix

y_true_bt = list(tok_utt_bt["validation"]["labels"])
y_pred_bt = preds_bt.tolist()

print("Predicted label counts:", Counter(y_pred_bt))
print("True label counts:", Counter(int(x) for x in y_true_bt))

print(classification_report(
    y_true_bt,
    y_pred_bt,
    target_names=[id2label[i] for i in range(4)],
    digits=4
))

print(confusion_matrix(y_true_bt, y_pred_bt))

Predicted label counts: Counter({0: 3316, 1: 2312, 2: 1748, 3: 693})
True label counts: Counter({0: 3125, 1: 2244, 2: 1775, 3: 925})
              precision    recall  f1-score   support

      inform     0.8067    0.8560    0.8306      3125
    question     0.9265    0.9545    0.9403      2244
   directive     0.7695    0.7577    0.7636      1775
  commissive     0.6176    0.4627    0.5290       925

    accuracy                         0.8167      8069
   macro avg     0.7801    0.7577    0.7659      8069
weighted avg     0.8101    0.8167    0.8118      8069

[[2675   46  213  191]
 [  11 2142   86    5]
 [ 257  104 1345   69]
 [ 373   20  104  428]]


In [28]:
# Emotion labels
emotion_id2label = {
    0: "no emotion",
    1: "anger",
    2: "disgust",
    3: "fear",
    4: "happiness",
    5: "sadness",
    6: "surprise"
}

emotion_label2id = {v: k for k, v in emotion_id2label.items()}

for split_name in ["train", "validation", "test"]:
    counts = pd.Series(flat_ds[split_name]["emotion"]).value_counts().sort_index()
    print(f"\n{split_name.upper()} emotion distribution:")
    for i, c in counts.items():
        print(f"{emotion_id2label[i]:>12}: {c}")


TRAIN emotion distribution:
  no emotion: 72143
       anger: 827
     disgust: 303
        fear: 146
   happiness: 11182
     sadness: 969
    surprise: 1600

VALIDATION emotion distribution:
  no emotion: 7108
       anger: 77
     disgust: 3
        fear: 11
   happiness: 684
     sadness: 79
    surprise: 107

TEST emotion distribution:
  no emotion: 6321
       anger: 118
     disgust: 47
        fear: 17
   happiness: 1019
     sadness: 102
    surprise: 116


In [29]:
# Prepare utterance-level emotion dataset
tok_emotion_bt = flat_ds.map(
    lambda batch: tokenizer_bt(
        batch["utterance"],
        truncation=True,
        max_length=96
    ),
    batched=True
)

tok_emotion_bt = tok_emotion_bt.remove_columns(
    [c for c in tok_emotion_bt["train"].column_names if c not in ["input_ids", "attention_mask", "emotion"]]
)

tok_emotion_bt = tok_emotion_bt.rename_column("emotion", "labels")
tok_emotion_bt.set_format("torch")

print(tok_emotion_bt)
print(tok_emotion_bt["train"][0])

Map:   0%|          | 0/87170 [00:00<?, ? examples/s]

Map:   0%|          | 0/8069 [00:00<?, ? examples/s]

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 87170
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 8069
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 7740
    })
})
{'labels': tensor(0), 'input_ids': tensor([    0,  2204,     7,  4083,     7,    84,    62,   117,    19,    11,
          433, 11255,   177,  1434,    21,     2]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}


In [30]:
# BERTweet emotion classifier
model_emotion_bt = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_bt,
    num_labels=7,
    id2label=emotion_id2label,
    label2id=emotion_label2id
).to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initi

In [31]:
# Metrics for emotion classifier
def compute_metrics_emotion(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

trainer_emotion_bt = Trainer(
    model=model_emotion_bt,
    args=training_args,
    train_dataset=tok_emotion_bt["train"],
    eval_dataset=tok_emotion_bt["validation"],
    data_collator=data_collator_bt,
    compute_metrics=compute_metrics_emotion
)

trainer_emotion_bt.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.414480,0.256788,0.902838,0.348675
2,0.331353,0.279929,0.889082,0.375726
3,0.273110,0.272149,0.899492,0.429555


TrainOutput(global_step=16347, training_loss=0.33964762504205664, metrics={'train_runtime': 541.9606, 'train_samples_per_second': 482.526, 'train_steps_per_second': 30.163, 'total_flos': 5564984916210600.0, 'train_loss': 0.33964762504205664, 'epoch': 3.0})

In [32]:
# Validation performance: emotion classifier
pred_output_emotion_bt = trainer_emotion_bt.predict(tok_emotion_bt["validation"])
preds_emotion_bt = pred_output_emotion_bt.predictions.argmax(axis=-1)
y_true_emotion = list(tok_emotion_bt["validation"]["labels"])

print(classification_report(
    y_true_emotion,
    preds_emotion_bt,
    target_names=[emotion_id2label[i] for i in range(7)],
    digits=3
))

              precision    recall  f1-score   support

  no emotion      0.953     0.938     0.945      7108
       anger      0.640     0.416     0.504        77
     disgust      0.000     0.000     0.000         3
        fear      1.000     0.091     0.167        11
   happiness      0.565     0.718     0.632       684
     sadness      0.455     0.190     0.268        79
    surprise      0.478     0.505     0.491       107

    accuracy                          0.899      8069
   macro avg      0.584     0.408     0.430      8069
weighted avg      0.905     0.899     0.900      8069



In [33]:
# Confusion matrix: emotion classifier
cm_emotion = confusion_matrix(y_true_emotion, preds_emotion_bt)
cm_emotion_df = pd.DataFrame(
    cm_emotion,
    index=[emotion_id2label[i] for i in range(7)],
    columns=[emotion_id2label[i] for i in range(7)]
)

cm_emotion_df

,no emotion,anger,disgust,fear,happiness,sadness,surprise
no emotion,6665,10,5,0,366,15,47
anger,36,32,1,0,2,2,4
disgust,1,1,0,0,0,0,1
fear,5,1,1,1,1,1,1
happiness,187,0,0,0,491,0,6
sadness,58,4,1,0,1,15,0
surprise,43,2,0,0,8,0,54


In [34]:
act_save_dir = "./act_bertweet_model"
emotion_save_dir = "./emotion_bertweet_model"

trainer_utt_bt.save_model(act_save_dir)
tokenizer_bt.save_pretrained(act_save_dir)

trainer_emotion_bt.save_model(emotion_save_dir)
tokenizer_bt.save_pretrained(emotion_save_dir)

print("Saved act model to:", act_save_dir)
print("Saved emotion model to:", emotion_save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved act model to: ./act_bertweet_model
Saved emotion model to: ./emotion_bertweet_model


In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
act_save_dir = "/content/drive/MyDrive/act_bertweet_model"
emotion_save_dir = "/content/drive/MyDrive/emotion_bertweet_model"

trainer_utt_bt.save_model(act_save_dir)
tokenizer_bt.save_pretrained(act_save_dir)

trainer_emotion_bt.save_model(emotion_save_dir)
tokenizer_bt.save_pretrained(emotion_save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/emotion_bertweet_model/tokenizer_config.json',
 '/content/drive/MyDrive/emotion_bertweet_model/vocab.txt',
 '/content/drive/MyDrive/emotion_bertweet_model/bpe.codes',
 '/content/drive/MyDrive/emotion_bertweet_model/added_tokens.json')